# 05 · Diagnose: are the `<sum>` tokens used?

| Setting | Value |
|---|---|
| Accelerator | **GPU T4 x2** (float16) or **L4** (bfloat16) — set `HW` |
| Internet | **ON** (pip, model download, cloning the 4 eval repos) |
| Attach | code (`scopesum-code`) unless `REPO_URL`; **data** from 01; the **run outputs** of 02 |
| Expected runtime | ~3 forward passes per window per variant with eager attention; ~10–15 min per variant at 60 files |

Per variant (`scopesum/diagnose.py`): loss with `<sum>` hidden from raw queries (sliding layers / all layers), attention share on `<sum>` / BOS / far raw tokens per layer type, and loss by distance from the start of the token's scope. Output: `/kaggle/working/scopesum_diag/<variant>.json` and `diagnose_summary.csv`.

In [ ]:
HW = "t4"            # "t4" (float16; Kaggle GPU T4 x2) or "l4" (bfloat16)
DATA_DIR = "/kaggle/input/scopesum-data/scopesum_data"   # auto-detected below if it does not exist
VARIANTS = ["A", "B", "C", "D", "E", "F", "G"]   # evaluated in this order; variants without a checkpoint are skipped
CHECKPOINTS = {}     # variant -> checkpoint dir; {} = auto-detect .../variant_X/final in the attached inputs
EVAL_OVERRIDES = {"max_eval_files": 60}  # long-range windows used for the diagnostics
N_ATTENTION = 16     # windows with attention statistics (eager attention, per variant)


REPO_URL = "https://github.com/loaferct/scopesum.git"  # private repo: needs the Kaggle Secret GITHUB_TOKEN; "" = use an attached code dataset
CODE_DATASET_DIR = "/kaggle/input"
import glob, os, shutil, subprocess, sys
CODE_DIR = "/kaggle/working/scopesum"

def clone_repo(url, dest, secret_name="GITHUB_TOKEN"):
    """git clone; uses a GitHub token from Kaggle Secrets for private repos (never printed)."""
    auth_url = url
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret(secret_name)
        auth_url = url.replace("https://", f"https://x-access-token:{token}@")
    except Exception as e:
        print(f"No Kaggle Secret {secret_name!r} ({type(e).__name__}); trying an anonymous clone")
    r = subprocess.run(["git", "clone", "--depth", "1", auth_url, dest], capture_output=True, text=True)
    if r.returncode != 0:
        print("git clone failed:", r.stderr.replace(auth_url, url)[-500:])
        return False
    subprocess.run(["git", "-C", dest, "remote", "set-url", "origin", url])  # drop the token from .git/config
    print("cloned", url, subprocess.run(["git", "-C", dest, "log", "-1", "--oneline"], capture_output=True, text=True).stdout)
    return True

if not os.path.exists(os.path.join(CODE_DIR, "scopesum", "__init__.py")):
    if not (REPO_URL and clone_repo(REPO_URL, CODE_DIR)):
        hits = [h for h in glob.glob(CODE_DATASET_DIR + "/**/scopesum/__init__.py", recursive=True) if "/tests/" not in h]
        assert hits, "Attach the scopesum code dataset or set REPO_URL"
        shutil.copytree(os.path.dirname(os.path.dirname(hits[0])), CODE_DIR)
os.chdir(CODE_DIR)
sys.path.insert(0, CODE_DIR)

In [ ]:
!pip uninstall -y -q torchao  # Kaggle ships torchao 0.10; peft>=0.21 refuses it and we do not use torchao
!pip install -q -r requirements.txt

In [ ]:
import json
from scopesum.config import load_config
from scopesum.data import load_stats, load_tokenizer, resolve_model_path, resolve_sum_token_id

def cfg_for(v):
    c = load_config("configs/base.yaml", f"configs/variant_{v}.yaml", f"configs/hw_{HW}.yaml",
                    overrides={"data": {"out_dir": DATA_DIR}, "eval": dict(EVAL_OVERRIDES)})
    if v == "B":
        c.variant.chunk_size = int(stats["chunk_size"])  # G sets its own K in variant_G.yaml
    return c

if not os.path.exists(os.path.join(DATA_DIR, "stats.json")):  # only stats.json is needed here (K, token ids)
    found = [p for p in glob.glob("/kaggle/input/**/stats.json", recursive=True) if os.path.basename(os.path.dirname(p)) == "scopesum_data"]
    print("DATA_DIR has no prepared data; found:", found)
    DATA_DIR = os.path.dirname(found[0])
stats = load_stats(DATA_DIR)
K = int(stats["chunk_size"])
if not CHECKPOINTS:
    for v in VARIANTS:
        hits = sorted(glob.glob(f"/kaggle/input/**/variant_{v}/final/adapter/adapter_config.json", recursive=True))
        if hits:
            CHECKPOINTS[v] = os.path.dirname(os.path.dirname(hits[0]))
print("checkpoints:", json.dumps(CHECKPOINTS, indent=1))
cfg = cfg_for("C")
OUT = "/kaggle/working/scopesum_diag"
os.makedirs(OUT, exist_ok=True)
path = resolve_model_path(cfg)
tok = load_tokenizer(cfg, path)
sum_id = resolve_sum_token_id(tok, cfg)
assert sum_id == stats["sum_id"]
print("K =", K, "| sum_id =", sum_id)

In [ ]:
from collections import Counter
from scopesum.eval_longrange import clone_eval_repos, find_eval_repo_files, prepare_long_range

root = clone_eval_repos(cfg.eval.eval_repo_urls, cfg.eval.eval_repos_dir)
eval_files = find_eval_repo_files(root, cfg.data.excluded_repos, cfg.eval.max_eval_files)
print(len(eval_files), "eval files;", Counter(next((r for r in cfg.data.excluded_repos if f"/{r}/" in p.lower()), "?") for p in eval_files))
lr_items = prepare_long_range(cfg, tok, eval_files)
print(len(lr_items), "windows;", sum(len(i["targets"]) for i in lr_items), "long-range target tokens")

In [ ]:
import gc, time
import torch
from scopesum.diagnose import diagnose_variant
from scopesum.eval_longrange import save_json
from scopesum.model import load_for_eval

for v, ck in CHECKPOINTS.items():
    out_path = os.path.join(OUT, f"{v}.json")
    if os.path.exists(out_path):
        print(f"[{v}] already done -> {out_path}"); continue
    cv = cfg_for(v)
    cv.model.attn_implementation = "eager"  # attention weights for the attention-share statistics
    t0 = time.time()
    lm = load_for_eval(cv, path, sum_id, ck)
    chunk = cv.variant.chunk_size if cv.variant.segment_mode == "fixed" else None
    res = diagnose_variant(lm, lr_items, tok, chunk, n_attention=N_ATTENTION)
    res["checkpoint"], res["minutes"] = ck, (time.time() - t0) / 60
    save_json(res, out_path)
    print(v, json.dumps({"loss": res["loss"], "delta": res.get("delta"), "attention": res["attention"]}, indent=1))
    del lm; gc.collect(); torch.cuda.empty_cache()

In [ ]:
import pandas as pd
rows, offs = [], []
for p in sorted(glob.glob(os.path.join(OUT, "*.json"))):
    r = json.load(open(p))
    for cond, d in r["loss"].items():
        row = {"variant": r["variant"], "condition": cond, **{f"loss_{k}": x["loss"] for k, x in d.items()}}
        for lt, a in r.get("attention", {}).items():
            if cond == "normal":
                row.update({f"{lt}_{k}": x for k, x in a.items()})
        rows.append(row)
    for cond, d in r["by_scope_offset"].items():
        offs += [{"variant": r["variant"], "condition": cond, "bucket": b, "loss": x["loss"], "n": x["n"]} for b, x in d.items()]
summary, by_offset = pd.DataFrame(rows), pd.DataFrame(offs)
summary.to_csv(os.path.join(OUT, "diagnose_summary.csv"), index=False)
by_offset.to_csv(os.path.join(OUT, "diagnose_by_offset.csv"), index=False)
print(summary.to_string(index=False))
print(by_offset[by_offset.condition == "normal"].pivot(index="variant", columns="bucket", values="loss").to_string())